In [32]:
import pandas as pd
import numpy as np
import os

# Load data directly from parquet files
data_dir = 'completejourney_py/completejourney_py/data'

transactions = pd.read_parquet(os.path.join(data_dir, 'transactions.parquet'))
demographics = pd.read_parquet(os.path.join(data_dir, 'demographics.parquet'))
products = pd.read_parquet(os.path.join(data_dir, 'products.parquet'))
campaigns = pd.read_parquet(os.path.join(data_dir, 'campaigns.parquet'))
campaign_descriptions = pd.read_parquet(os.path.join(data_dir, 'campaign_descriptions.parquet'))
promotions = pd.read_parquet(os.path.join(data_dir, 'promotions.parquet'))
#coupons = pd.read_parquet(os.path.join(data_dir, 'coupons.parquet'))
#coupon_redemptions = pd.read_parquet(os.path.join(data_dir, 'coupon_redemptions.parquet'))

## Demographics

In [43]:
demographics.head()

,household_id,age,income,home_ownership,marital_status,household_size,household_comp,kids_count
0,1,65+,35-49K,Homeowner,Married,2,2 Adults No Kids,0
1,1001,45-54,50-74K,Homeowner,Unmarried,1,1 Adult No Kids,0
2,1003,35-44,25-34K,None,Unmarried,1,1 Adult No Kids,0
3,1004,25-34,15-24K,None,Unmarried,1,1 Adult No Kids,0
4,101,45-54,Under 15K,Homeowner,Married,4,2 Adults Kids,2


In [44]:
demographics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   household_id    801 non-null    int64 
 1   age             801 non-null    object
 2   income          801 non-null    object
 3   home_ownership  568 non-null    object
 4   marital_status  664 non-null    object
 5   household_size  801 non-null    object
 6   household_comp  801 non-null    object
 7   kids_count      801 non-null    object
dtypes: int64(1), object(7)
memory usage: 50.2+ KB


### Demographics Data Cleaning

In [45]:
missing_counts = demographics.isnull().sum()
print(missing_counts)

household_id        0
age                 0
income              0
home_ownership    233
marital_status    137
household_size      0
household_comp      0
kids_count          0
dtype: int64


Considering the amount of missing values in home_ownership and marital status, and the fact that they most likely don't have much informative data on customer churn, we remove these columns from the dataset. 

In [46]:
demographics = demographics.drop(['home_ownership', 'marital_status'], axis=1)

In [47]:
ordinal_categorical_demographics = ['age', 'income', 'household_size', 'kids_count']
nominal_categorical_demographics = ['household_comp']

In [49]:
demographics.duplicated().sum()

np.int64(0)

In [52]:
for col in ordinal_categorical_demographics:
    print(f"\n{col.upper()} - Unique Values:")
    print(demographics[col].unique())

print("HOUSEHOLD_COMP - Unique Values:")
print(demographics['household_comp'].unique())


AGE - Unique Values:
['65+' '45-54' '35-44' '25-34' '55-64' '19-24']

INCOME - Unique Values:
['35-49K' '50-74K' '25-34K' '15-24K' 'Under 15K' '75-99K' '100-124K'
 '125-149K' '150-174K' '250K+' '175-199K' '200-249K']

HOUSEHOLD_SIZE - Unique Values:
['2' '1' '4' '5+' '3']

KIDS_COUNT - Unique Values:
['0' '2' '3+' '1']
HOUSEHOLD_COMP - Unique Values:
['2 Adults No Kids' '1 Adult No Kids' '2 Adults Kids' '1 Adult Kids']


In [48]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (valid + test combined)
demographics_index_train, demographics_index_temp = train_test_split(
    np.array(demographics.index),  
    train_size=0.7,
    random_state=5
)

# Second split: Split the 30% temp into 50% valid, 50% test
demographics_index_valid, demographics_index_test = train_test_split(
    demographics_index_temp,
    train_size=0.5,
    random_state=5
)

demographics_train = demographics.loc[demographics_index_train,:].copy()
demographics_valid = demographics.loc[demographics_index_valid,:].copy()
demographics_test = demographics.loc[demographics_index_test,:].copy()

In [ ]:
transactions['transaction_timestamp'] = pd.to_datetime(transactions['transaction_timestamp'])
print(transactions['transaction_timestamp'])

0         2017-01-01 11:53:26
1         2017-01-01 12:10:28
2         2017-01-01 12:26:30
3         2017-01-01 12:30:27
4         2017-01-01 12:30:27
                  ...        
1469302   2018-01-01 03:50:03
1469303   2018-01-01 04:01:20
1469304   2018-01-01 04:01:20
1469305   2018-01-01 04:01:20
1469306   2018-01-01 04:01:20
Name: transaction_timestamp, Length: 1469307, dtype: datetime64[ns]
